In [1]:
import torch
import math
import random
import matplotlib.pyplot as plt
import torch.nn.functional as F
torch.manual_seed(0)

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
DATA_PATH = '/content/drive/MyDrive/data/cifar10.pt'
d = torch.load(DATA_PATH, map_location=device)
print(device)

cuda


In [3]:
# @title
# -- Helper functions --

# Plot view pairs in grid WIP
def plot_grid(views_list):
  w = h = 32
  fig = plt.figure(figsize=(7, 14))
  columns = len(views_list)
  rows = len(views_list[0])
  for i in range(columns*rows):
      img = views_list[i % columns][i // columns]
      fig.add_subplot(rows, columns, i+1)
      plt.imshow(img.cpu().permute(1,2,0), interpolation='nearest')
      plt.axis('off')
  plt.subplots_adjust(wspace=0, hspace=0)
  plt.show()


# Other helpers made by Claude
# (batch image transforms not supported by torchvision)
LUMA = torch.tensor([0.2989, 0.5870, 0.1140]).view(1, 3, 1, 1)

def _rand(B, lo, hi, device):
    return torch.empty(B, 1, 1, 1, device=device).uniform_(lo, hi)

def _gray(x):                                  # (B,3,H,W) -> (B,3,H,W)
    return (x * LUMA.to(x.device)).sum(1, keepdim=True).expand_as(x)

def brightness_(x, lo=0.6, hi=1.4):
    x.mul_(_rand(x.size(0), lo, hi, x.device))
    return x

def contrast_(x, lo=0.6, hi=1.4):
    f = _rand(x.size(0), lo, hi, x.device)
    mean = _gray(x).mean((1, 2, 3), keepdim=True)   # scalar per image
    x.lerp_(mean, 1 - f)
    return x

def saturation_(x, lo=0.6, hi=1.4):
    f = _rand(x.size(0), lo, hi, x.device)
    x.lerp_(_gray(x), 1 - f)
    return x

def hue_(x, amount=0.1):
    # cheap approximation: rotate the two chroma axes about luma
    f = _rand(x.size(0), -amount, amount, x.device) * 2 * 3.141592653589793
    c, s = torch.cos(f), torch.sin(f)
    g = _gray(x)[:, :1]
    r, gr, b = x[:, 0:1] - g, x[:, 1:2] - g, x[:, 2:3] - g
    x[:, 0:1] = g + c * r - s * b
    x[:, 1:2] = g + gr                              # green anchored
    x[:, 2:3] = g + s * r + c * b
    return x

def hflip_(x, p=0.5):
    m = torch.rand(x.size(0), device=x.device) < p
    x[m] = x[m].flip(-1)
    return x

def greyscale_(x, p=0.2):
    m = torch.rand(x.size(0), device=x.device) < p
    x[m] = _gray(x[m])
    return x

def rrc(x, out=32, scale=(0.4, 1.0), ratio=(3/4, 4/3)):
    B, dev = x.size(0), x.device
    area = torch.empty(B, device=dev).uniform_(*scale)
    logr = torch.empty(B, device=dev).uniform_(math.log(ratio[0]), math.log(ratio[1]))
    ar = torch.exp(logr)
    w = (area * ar).sqrt().clamp(max=1.0)      # crop width as fraction of image
    h = (area / ar).sqrt().clamp(max=1.0)
    cx = torch.rand(B, device=dev) * (1 - w) * 2 - (1 - w)   # centre in [-1,1] coords
    cy = torch.rand(B, device=dev) * (1 - h) * 2 - (1 - h)
    theta = torch.zeros(B, 2, 3, device=dev)
    theta[:, 0, 0], theta[:, 0, 2] = w, cx
    theta[:, 1, 1], theta[:, 1, 2] = h, cy
    grid = F.affine_grid(theta, (B, 3, out, out), align_corners=False)
    return F.grid_sample(x, grid, mode='bilinear', align_corners=False)

## Plan:
* Split into patches of 4x4 and pass each patch through a single learnable linear layer of 4x4x3->192 to get 64 tokens
* For each patch pass its index (encoded as one hot) into another learnable linear layer 64->192 which gives us a position embedding
* Add position embeddings to the tokens to get input into the transformer

Now pass them into a transformer block:

* First build attention by creating 3 learnable matrixes for every one of the $h$ heads $W_k$, $W_q$, $W_v$, each of size $192 \times (192/h)$.
* Apply KQV to each element for each head.
* Using matrix multiplications to do this in parallell: loop through every token and apply $W_q$ to it after layerNorm. For each one of those loop through every token again and apply $W_k$ to it after layerNorm. Dot the two outputs to get a weight, divide weight by $\sqrt{192/h}$ to keep at unit variance; then softmax across all the weights in the loop. Multiply each weight by the relevant second tokens $W_v$. Sum across all tokens in inner loop to get 'embedding shift' vector. Repeat for every token in outer loop.
* Concantente all of the heads ouputs together (to restore each embeddings length) and apply the learnable output matrix $W_O$ to it
* Add each result to the original vector i.e. $x = x + f(norm(x))$
* Pass each layerNorm'd embedding through an MLP and add onto original

Pass them through 4 more transformer blocks to get an output sequence, then layerNorm and global average pool to add together all the different meanings from the image.
The output is the picture embedded. ViT complete.

----
Choose h = 6

Use standard 4 layer MLP `Linear(192, 768) --> GELU --> Linear(768, 192)`

Combine all 3 matrixes and all h heads into one matrix multiplication operation



In [4]:
# @title
# Takes in a batch of images and returns a batch of tokenised images.
# Each image is split into patches and each patch
# is flattened and transformed into a 192dim token.
# Positional embeddings are then added on.
class Tokeniser(torch.nn.Module):

  def __init__(self, IMAGE_SIZE, OUT_DIMS, PATCH_SIZE):
    super().__init__()
    self.TOKEN_COUNT = (IMAGE_SIZE // PATCH_SIZE)**2
    self.patch_embedder = torch.nn.Conv2d(3, OUT_DIMS, PATCH_SIZE, stride=PATCH_SIZE)
    self.pos_embeddings = torch.nn.parameter.Parameter(torch.zeros(self.TOKEN_COUNT, OUT_DIMS)) # TODO: work out good std and switch to randomised

  def forward(self, X):
    # Convert from (N, 192, 8, 8) to (N, 64, 192)
    tokens = self.patch_embedder(X).flatten(2).transpose(1,2)
    tokens = tokens + self.pos_embeddings
    return tokens


class Attention(torch.nn.Module):

  def __init__(self, N_HEADS, DIMS):
    super().__init__()
    self.W_KQV = torch.nn.Linear(DIMS, DIMS*3, bias=False)
    self.W_O = torch.nn.Linear(DIMS, DIMS)
    self.N_HEADS = N_HEADS
    self.DIMS = DIMS

  def forward(self, X):
    # Compute KQV all at once, for every element in the batch
    KQV = self.W_KQV(X)

    # Split into separate K, Q and V
    K, Q, V = KQV.split(self.DIMS, dim=-1)

    # Split across heads into (N, Tokens, Heads, Elements)
    d_k = self.DIMS // self.N_HEADS
    shape = K.shape[:2] + (self.N_HEADS, d_k)

    K = K.reshape(shape)
    Q = Q.reshape(shape)
    V = V.reshape(shape)

    # Also permute to (N, Heads, Tokens, Elements) for matmuls
    K = K.permute((0,2,1,3))
    Q = Q.permute((0,2,1,3))
    V = V.permute((0,2,1,3))

    weights = Q @ K.mT
    weights = weights / math.sqrt(d_k)
    weights = F.softmax(weights, dim=-1)

    embeddings = weights @ V
    # (N, heads, tokens, head_size) e.g. (N, 6, 64, 32)
    embeddings_shape = shape[:2] + (self.DIMS, )
    # (N, 6, 64, 32) --> (N, 64, 6, 32) --> (N, 64, 192)
    embeddings = embeddings.transpose(1, 2).contiguous().reshape(embeddings_shape)
    embeddings = self.W_O(embeddings)

    return embeddings


class Transformer_Block(torch.nn.Module):

  def __init__(self, DIMS, N_HEADS):
    super().__init__()
    self.norm_a = torch.nn.LayerNorm(DIMS)
    self.norm_b = torch.nn.LayerNorm(DIMS)
    self.attention = Attention(N_HEADS, DIMS)
    self.MLP = torch.nn.Sequential(
        torch.nn.Linear(DIMS, 4 * DIMS),
        torch.nn.GELU(),
        torch.nn.Linear(4 * DIMS, DIMS)
    )

  def forward(self, X):
    X = X + self.attention(self.norm_a(X))
    X = X + self.MLP(self.norm_b(X))
    return X


class ViT(torch.nn.Module):

  def __init__(self, IMAGE_SIZE, PATCH_SIZE=4, DIMS=192, N_HEADS=6):
    super().__init__()
    self.tokeniser = Tokeniser(IMAGE_SIZE, DIMS, PATCH_SIZE)
    self.transformer_blocks = torch.nn.Sequential(
      Transformer_Block(DIMS, N_HEADS),
      Transformer_Block(DIMS, N_HEADS),
      Transformer_Block(DIMS, N_HEADS),
      Transformer_Block(DIMS, N_HEADS),
      Transformer_Block(DIMS, N_HEADS)
    )
    self.norm = torch.nn.LayerNorm(DIMS)

  def forward(self, X):
    tokens    = self.tokeniser(X)
    patch_embeddings = self.transformer_blocks(tokens)
    patch_embeddings = self.norm(patch_embeddings)
    image_embedding  = patch_embeddings.mean(dim=1)

    return image_embedding


In [5]:
def create_view(x):
  view = rrc(x)
  if random.random() < 0.8: # This is batch level not sample level - TODO?
    brightness_(view)
    contrast_(view)
    saturation_(view)
    hue_(view)
  hflip_(view)
  greyscale_(view)
  return view.clamp_(0, 1)

In [6]:
@torch.inference_mode()
def evaluate(Y_pred, Y):
  N = len(Y)
  correct = (Y_pred.argmax(1) == Y).sum().item()

  return correct

In [7]:
data_perm = torch.randperm(50000)
train_idx, val_idx = data_perm[:45_000], data_perm[45_000:] # Split into 45K/5K

train_data_X, train_data_Y = d['xtr'][train_idx].float() / 255, d['ytr'][train_idx]
val_data_X, val_data_Y = d['xtr'][val_idx].float() / 255, d['ytr'][val_idx]
test_data_X, test_data_Y = d['xte'].float() / 255, d['yte']

mean = train_data_X.mean((0, 2, 3), keepdim=True)
std  = train_data_X.std((0, 2, 3), keepdim=True)

In [8]:
BATCH_SIZE = 128
IMAGE_SIZE = train_data_X.shape[-1]
N = len(train_data_X)

model = ViT(32, 4, 192, 6).to(device)
_classifier = torch.nn.Linear(192, 10).to(device)
loss_fn = torch.nn.CrossEntropyLoss().to(device)
optimiser = torch.optim.AdamW(list(model.parameters())+list(_classifier.parameters()), lr=1E-3)

In [9]:
for epoch in range(20):
  model.train()
  perm = torch.randperm(N, device=device)
  loss_value = -1

  for i in range(0, N - N % BATCH_SIZE, BATCH_SIZE):
    idx = perm[i:i+BATCH_SIZE]
    X, Y = (train_data_X[idx] - mean) / std, train_data_Y[idx] # Temp, revert normalisation of X

    # Generate views by (applying transforms)
    #views_a = create_view(X)
    #views_b = create_view(X)

    # Normalise
    #views_a = (views_a - mean) / std
    #views_b = (views_b - mean) / std

    # ---- Temp test/ pre cleanup

    # Tokenise
    embedding = model(X)
    Y_pred = _classifier(embedding)

    loss = loss_fn(Y_pred.float(), Y)
    optimiser.zero_grad()
    loss.backward()
    optimiser.step()

    loss_value = loss.item()
    # ----

  with torch.inference_mode():
    model.eval()
    val_embedding = model((val_data_X - mean) / std)
    Y_val_pred = _classifier(val_embedding)
    correct = evaluate(Y_val_pred, val_data_Y)

  print(f'Val for epoch {epoch:<3} | {correct / 5000:.4f} | {loss_value}')

Val for epoch 0   | 0.4164 | 1.583853840827942
Val for epoch 1   | 0.5188 | 1.4194378852844238
Val for epoch 2   | 0.5476 | 1.3279341459274292
Val for epoch 3   | 0.5462 | 1.2862648963928223
Val for epoch 4   | 0.5736 | 1.1145836114883423
Val for epoch 5   | 0.5946 | 1.04257333278656
Val for epoch 6   | 0.6046 | 1.1417839527130127
Val for epoch 7   | 0.6096 | 1.0212432146072388
Val for epoch 8   | 0.6318 | 0.9606052041053772
Val for epoch 9   | 0.6380 | 0.8501068353652954
Val for epoch 10  | 0.6372 | 0.9720690846443176
Val for epoch 11  | 0.6464 | 0.7619501352310181
Val for epoch 12  | 0.6566 | 0.8557320237159729
Val for epoch 13  | 0.6564 | 0.9667940139770508
Val for epoch 14  | 0.6774 | 0.6958658695220947
Val for epoch 15  | 0.6728 | 0.6921905279159546
Val for epoch 16  | 0.6672 | 0.8328537940979004
Val for epoch 17  | 0.6716 | 0.610791802406311
Val for epoch 18  | 0.6796 | 0.6580579280853271
Val for epoch 19  | 0.6938 | 0.589093804359436


In [10]:
with torch.inference_mode():
  model.eval()
  test_embedding = model((test_data_X - mean) / std)
  Y_test_pred = _classifier(test_embedding)
  correct = evaluate(Y_test_pred, test_data_Y)
  print(f'Correct: {correct / len(test_data_Y):.4f}')

Correct: 0.6856


Cleanup TODOs:
- Make a proper eval function
- Move all the helper functions out into another file
- Standadise dataset on init
- Switch position embeddings to randomised